# Question Sampling for Human Study (Control Questions)

Goal: select a manageable subset of questions (~100–150) from the 1K VQA control set
that are most informative for human data collection.

## Sampling Criteria

| Tier | Criterion | Rationale |
|------|-----------|-----------|
| A | **Overconfident & Wrong** | High model confidence + low accuracy = hallucination signal |
| B | **Answer Flip across variants** | Model changes answer when question is weakened = genuine linguistic sensitivity |
| C | **Cross-model consensus wrong** | All models give same wrong answer = shared corpus prior |
| D | **High degradation** | Large accuracy drop from `question` → `subject_ablated` = shortcut dependent |
| E | **Stratified by answer type** | Ensure coverage of yes/no, number, color, location, other |

Target: ~30 questions per tier, stratified by answer type, final deduplicated set of ~100–150.

In [1]:
import json
import pandas as pd
import numpy as np
import os
import sys
sys.path.insert(0, '/home/david/Desktop/yuna/HPA/analysis')
from utils.vqa import VQAAnswerMapper, vqa_accuracy

RESULTS_DIR   = '/home/david/Desktop/yuna/HPA/evaluation/logits/pretrained'
SEMANTICS_PATH = '/home/david/Desktop/yuna/HPA/dataset/vqa/vqa1k_semantics.jsonl'

mapper = VQAAnswerMapper()
CONTROL_TYPES = ['question', 'deictic_removed', 'object_removed', 'weaker_object', 'subject_ablated']

# Load entity grouping from semantics
sem_df = pd.read_json(SEMANTICS_PATH, lines=True)[['question_id', 'ent', 'w', 'op', 'attr', 'ans']]
qid_to_ent = sem_df.set_index('question_id')['ent'].to_dict()
print("Entity categories:", sorted(sem_df['ent'].dropna().unique()))


Entity categories: ['animal', 'food', 'object', 'other', 'person', 'place', 'product', 'text', 'vehicle']


In [2]:
def load_control(model, dataset='vqa_1k_control'):
    filepath = f'{RESULTS_DIR}/{model}/{dataset}.jsonl'
    if not os.path.exists(filepath):
        return []
    rows = []
    with open(filepath) as f:
        for line in f:
            ex = json.loads(line)
            if 'generated_logits' not in ex:
                continue
            gt_answers = mapper.get_answers(ex['question_id'])
            output_key = 'generated_answers' if 'generated_answers' in ex else 'answers'
            ent = qid_to_ent.get(ex['question_id'], 'other')
            for ct in ex['generated_logits']:
                logits = ex['generated_logits'][ct]['content']
                token_probs = np.exp([t['logprob'] for t in logits])
                confidence = float(token_probs.mean())
                output = ex[output_key].get(ct, '')
                accuracy = vqa_accuracy(output, gt_answers) if gt_answers else None
                rows.append({
                    'model': model,
                    'question_id': ex['question_id'],
                    'control_type': ct,
                    'question_text': ex.get(ct, ''),
                    'output': output,
                    'confidence': confidence,
                    'accuracy': accuracy,
                    'ent': ent,
                })
    return rows

models = [m for m in os.listdir(RESULTS_DIR)
          if os.path.exists(f'{RESULTS_DIR}/{m}/vqa_1k_control.jsonl')]
print('Models found:', models)

all_rows = []
for m in models:
    all_rows.extend(load_control(m))

df = pd.DataFrame(all_rows)
df['accuracy'] = pd.to_numeric(df['accuracy'], errors='coerce')
print(f'Loaded {len(df)} rows | {df["model"].nunique()} models | {df["question_id"].nunique()} questions')
print(f'\nEntity distribution:\n{df[df["control_type"]=="question"]["ent"].value_counts()}')


Models found: ['Qwen3-VL-32B-Instruct', 'llava-v1.6-vicuna-7b-hf', 'llava-v1.6-vicuna-13b-hf', 'llava-1.5-7b-hf', 'Qwen3-VL-4B-Instruct', 'llava-v1.6-mistral-7b-hf', 'Qwen3-VL-8B-Instruct']
   Loading VQA annotations from /home/david/Desktop/yuna/data/v2_mscoco_val2014_annotations.json...
   ✓ Loaded 214354 VQA annotations
Loaded 25215 rows | 6 models | 1000 questions

Entity distribution:
ent
object     1741
person     1168
animal      540
other       423
food        391
place       302
vehicle     261
text        131
product      86
Name: count, dtype: int64


In [ ]:
# Aggregate per (question_id, control_type) across models
agg = df.groupby(['question_id', 'control_type', 'ent']).agg(
    mean_accuracy=('accuracy', 'mean'),
    mean_confidence=('confidence', 'mean'),
    n_models=('model', 'nunique'),
    outputs=('output', lambda x: list(x)),
).reset_index()

# Per question: baseline (original question) and most-weakened (subject_ablated)
baseline = agg[agg['control_type'] == 'question'][['question_id', 'mean_accuracy', 'mean_confidence', 'ent']]
baseline.columns = ['question_id', 'acc_baseline', 'conf_baseline', 'ent']
ablated = agg[agg['control_type'] == 'subject_ablated'][['question_id', 'mean_accuracy', 'mean_confidence']]
ablated.columns = ['question_id', 'acc_ablated', 'conf_ablated']

qdf = baseline.merge(ablated, on='question_id')
qdf['acc_drop']  = qdf['acc_baseline']  - qdf['acc_ablated']
qdf['conf_drop'] = qdf['conf_baseline'] - qdf['conf_ablated']   # Tier E

# Tier A — overconfident & wrong
qdf['overconfident_wrong'] = (qdf['conf_baseline'] > 0.75) & (qdf['acc_baseline'] < 0.2)

# Tier B — answer flip across control variants
flip = df.groupby(['question_id', 'model']).apply(
    lambda g: g.set_index('control_type')['output'].reindex(CONTROL_TYPES).nunique(dropna=True)
).reset_index()
flip.columns = ['question_id', 'model', 'n_unique_outputs']
flip_agg = flip.groupby('question_id')['n_unique_outputs'].mean().reset_index()
flip_agg.columns = ['question_id', 'mean_output_changes']

# Tier C — majority of models wrong on baseline (relaxed: no requirement for same answer)
consensus = (
    df[df['control_type'] == 'question']
    .groupby('question_id')
    .apply(lambda g: (g['accuracy'] < 0.2).mean())
    .reset_index()
)
consensus.columns = ['question_id', 'frac_models_wrong']
consensus['consensus_wrong'] = consensus['frac_models_wrong'] >= 0.8

qdf = qdf.merge(flip_agg, on='question_id').merge(
    consensus[['question_id', 'consensus_wrong', 'frac_models_wrong']], on='question_id'
)
qdf.head()


In [ ]:
print('=== Criterion Counts ===')
print(f"A. Overconfident & Wrong (conf>0.75, acc<0.2):        {qdf['overconfident_wrong'].sum()}")
print(f"B. Answer flip (mean unique outputs > 2):             {(qdf['mean_output_changes'] > 2).sum()}")
print(f"C. Consensus wrong (≥80% models wrong on baseline):  {qdf['consensus_wrong'].sum()}")
print(f"D. High accuracy drop (acc_drop > 0.3):              {(qdf['acc_drop'] > 0.3).sum()}")
print(f"E. High confidence drop (conf_drop > 0.05):          {(qdf['conf_drop'] > 0.05).sum()}")
print(f"   — of which: confident→uncertain despite correct:  "
      f"{((qdf['conf_drop'] > 0.05) & (qdf['acc_baseline'] > 0.5)).sum()}")
print()
print('Entity distribution (all 1K questions):')
print(qdf['ent'].value_counts())

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(qdf['acc_drop'], qdf['conf_drop'], alpha=0.3, s=10)
axes[0].axhline(0.05, color='r', linestyle='--', label='conf_drop threshold')
axes[0].axvline(0.3, color='b', linestyle='--', label='acc_drop threshold')
axes[0].set_xlabel('Accuracy drop (question → subject_ablated)')
axes[0].set_ylabel('Confidence drop (question → subject_ablated)')
axes[0].set_title('Accuracy vs Confidence Drop')
axes[0].legend(fontsize=8)

axes[1].scatter(qdf['acc_baseline'], qdf['conf_drop'], alpha=0.3, s=10, c=qdf['conf_baseline'], cmap='RdYlGn')
axes[1].axhline(0.05, color='r', linestyle='--')
axes[1].set_xlabel('Baseline accuracy')
axes[1].set_ylabel('Confidence drop')
axes[1].set_title('Baseline Accuracy vs Confidence Drop\n(color = baseline confidence)')
plt.tight_layout()
plt.show()


In [ ]:
np.random.seed(42)

N_PER_TIER = 30

def stratified_sample(mask, n, ent_col='ent', seed=42):
    """Sample n questions from mask, stratified by entity group."""
    pool = qdf[mask].copy()
    if len(pool) == 0:
        print("  → empty pool, skipping")
        return []
    counts = pool[ent_col].value_counts(normalize=True)
    sampled = []
    for ent, frac in counts.items():
        k = max(1, round(n * frac))
        sub = pool[pool[ent_col] == ent]
        sampled.extend(sub.sample(min(k, len(sub)), random_state=seed)['question_id'].tolist())
    return list(set(sampled))[:n]

tier_A = stratified_sample(qdf['overconfident_wrong'],        N_PER_TIER)
tier_B = stratified_sample(qdf['mean_output_changes'] > 2,    N_PER_TIER)
tier_C = stratified_sample(qdf['consensus_wrong'],            N_PER_TIER)
tier_D = stratified_sample(qdf['acc_drop'] > 0.3,            N_PER_TIER)
tier_E = stratified_sample(qdf['conf_drop'] > 0.05,          N_PER_TIER)

# Combine, deduplicate, label tiers
all_sampled = {}
for qid in tier_A: all_sampled[qid] = all_sampled.get(qid, []) + ['A_overconfident_wrong']
for qid in tier_B: all_sampled[qid] = all_sampled.get(qid, []) + ['B_answer_flip']
for qid in tier_C: all_sampled[qid] = all_sampled.get(qid, []) + ['C_consensus_wrong']
for qid in tier_D: all_sampled[qid] = all_sampled.get(qid, []) + ['D_high_acc_drop']
for qid in tier_E: all_sampled[qid] = all_sampled.get(qid, []) + ['E_high_conf_drop']

sampled_df = pd.DataFrame([
    {'question_id': qid, 'tiers': '|'.join(tiers)}
    for qid, tiers in all_sampled.items()
])
sampled_df = sampled_df.merge(qdf, on='question_id')
sampled_df['n_tiers'] = sampled_df['tiers'].str.count(r'\|') + 1
sampled_df = sampled_df.sort_values('n_tiers', ascending=False)

print(f'Total unique questions selected: {len(sampled_df)}')
print(f'Questions in multiple tiers:     {(sampled_df["n_tiers"] > 1).sum()}')
print()
print('Entity distribution in sample:')
print(sampled_df['ent'].value_counts())
print()
print('Tier breakdown:')
for tier in ['A_overconfident_wrong', 'B_answer_flip', 'C_consensus_wrong', 'D_high_acc_drop', 'E_high_conf_drop']:
    print(f'  {tier}: {sampled_df["tiers"].str.contains(tier).sum()}')


In [6]:
# Attach question text and all model outputs for manual review
question_texts = df[df['control_type'] == 'question'][['question_id', 'question_text']].drop_duplicates('question_id')

outputs_pivot = df[
    (df['question_id'].isin(sampled_df['question_id'])) &
    (df['control_type'] == 'question')
].pivot_table(index='question_id', columns='model', values='output', aggfunc='first').reset_index()

review_df = sampled_df.merge(question_texts, on='question_id').merge(outputs_pivot, on='question_id')

display_cols = ['question_id', 'ent', 'tiers', 'n_tiers', 'acc_baseline', 'acc_drop',
                'conf_baseline', 'frac_models_wrong', 'question_text'] + list(outputs_pivot.columns[1:])
review_df[display_cols].head(20)


,question_id,ent,tiers,n_tiers,acc_baseline,acc_drop,conf_baseline,frac_models_wrong,question_text,Qwen3-VL-32B-Instruct,Qwen3-VL-8B-Instruct,llava-1.5-7b-hf,llava-v1.6-mistral-7b-hf,llava-v1.6-vicuna-13b-hf,llava-v1.6-vicuna-7b-hf
0,72096002,object,A_overconfident_wrong|B_answer_flip|C_consensu...,3,0.066667,0.066667,0.754867,0.8,Question: What part of the tablecloth flowers ...,NaN,none,Center,Flowers,Roses,Tablecloth
1,13769009,place,A_overconfident_wrong|B_answer_flip|C_consensu...,3,0.000000,0.000000,0.835658,1.0,Question: What is this bathroom missing? Answe...,NaN,toilet paper,Toilet paper,Toilet paper,Toilet paper,Toilet paper
2,346577000,person,B_answer_flip|D_high_degradation,2,1.000000,0.733333,0.959047,0.0,Question: What color is his shirt? Answer the ...,NaN,black,Black,Black,Black,Black
3,69959003,animal,B_answer_flip|D_high_degradation,2,1.000000,1.000000,0.872728,0.0,Question: What color is the dog? Answer the qu...,NaN,black and white,Black and white,Black and white,Black and white,Black and white
4,157581003,food,B_answer_flip|C_consensus_wrong,2,0.066667,-0.066667,0.663323,0.8,Question: How many bottles of while is there? ...,NaN,9,1,0,10,1
5,6712002,place,B_answer_flip|D_high_degradation,2,1.000000,0.866667,0.876250,0.0,Question: What color is the house? Answer the ...,NaN,blue,Blue,Blue,Blue,Blue
6,124601004,food,B_answer_flip|D_high_degradation,2,1.000000,1.000000,0.990837,0.0,Question: How many oranges are there? Answer t...,NaN,0,0,0,0,0
7,486576004,person,B_answer_flip|D_high_degradation,2,1.000000,1.000000,0.994135,0.0,Question: Is that a man or woman looking at th...,NaN,man,Man,Man,Man,Man
8,87052017,person,A_overconfident_wrong|C_consensus_wrong,2,0.066667,0.000000,0.864957,0.8,Question: What is the woman wearing on top? An...,NaN,sweatshirt,Sweater,Sweater,Sweater,Sweater
9,158279002,text,A_overconfident_wrong|C_consensus_wrong,2,0.000000,0.000000,0.886617,1.0,Question: What is the web address listed? Answ...,NaN,http://foodiebaker.com,Wwwfoodiebakercom,Wwwfoodiebakercom,Wwwfoodiebakercom,Wwwfoodiebakercom


In [7]:
# Export for manual review / human study design
out_path = '/home/david/Desktop/yuna/HPA/analysis/csv/sampled_control_questions.csv'
review_df.to_csv(out_path, index=False)
print(f'Saved {len(review_df)} questions to {out_path}')

# Summary by tier
print('\n=== Tier Breakdown ===')
for tier in ['A_overconfident_wrong', 'B_answer_flip', 'C_consensus_wrong', 'D_high_degradation']:
    n = sampled_df['tiers'].str.contains(tier).sum()
    print(f'  {tier}: {n} questions')

Saved 98 questions to /home/david/Desktop/yuna/HPA/analysis/csv/sampled_control_questions.csv

=== Tier Breakdown ===
  A_overconfident_wrong: 26 questions
  B_answer_flip: 30 questions
  C_consensus_wrong: 30 questions
  D_high_degradation: 30 questions
